# EvalForge — Complete Guide

EvalForge scores LLM outputs systematically. This notebook covers:

- EvalCase and EvalDataset
- ExactMatchScorer
- RegexScorer
- RougeScorer (ROUGE-L)
- LLMJudgeScorer (local, uses OpenAI)
- HostedScorer — pairwise and direct judge via hosted API
- run_eval and EvalReport
- compare — baseline vs candidate
- Real-world: evaluating a QA system

## Setup

In [ ]:
%pip install ag-evalforge openai -q

In [ ]:
import os
AG_API_KEY = os.environ['AG_API_KEY']
# OPENAI_API_KEY = os.environ['OPENAI_API_KEY']  # for LLMJudgeScorer

## 1. Building a dataset

In [ ]:
from ag_evalforge import EvalCase, EvalDataset

cases = EvalDataset(cases=[
    EvalCase(input='What is the capital of France?',
             expected='Paris',
             metadata={'category': 'geography'}),
    EvalCase(input='What is 12 × 12?',
             expected='144',
             metadata={'category': 'arithmetic'}),
    EvalCase(input='Who wrote Hamlet?',
             expected='William Shakespeare',
             metadata={'category': 'literature'}),
    EvalCase(input='What is the boiling point of water in Celsius?',
             expected='100',
             metadata={'category': 'science'}),
    EvalCase(input='Translate "hello" to Spanish.',
             expected='hola',
             metadata={'category': 'translation'}),
])

print(f'Dataset: {len(cases)} cases')
print(cases.cases[0])

## 2. ExactMatchScorer

Passes if the output exactly matches the expected (case-insensitive, stripped).

In [ ]:
from ag_evalforge import run_eval, ExactMatchScorer

outputs = ['Paris', '144', 'Shakespeare', '100 degrees Celsius', 'hola']

report = run_eval(cases, ExactMatchScorer(), outputs=outputs)
print(report.summary())
for r in report.results:
    icon = '✅' if r.passed else '❌'
    print(f'  {icon} expected={r.case.expected!r:25} got={r.output!r}')

## 3. RegexScorer

Passes if the output matches a regular expression — useful for format validation.

In [ ]:
from ag_evalforge import RegexScorer

# Check that outputs contain a 3-digit or 4-digit number (for arithmetic questions)
numeric_cases = EvalDataset(cases=[
    EvalCase(input='What is 12 × 12?',         expected='144'),
    EvalCase(input='What is 100 + 200?',        expected='300'),
    EvalCase(input='Square root of 144?',       expected='12'),
])

numeric_outputs = ['The answer is 144.', '300', 'twelve']

report = run_eval(numeric_cases, RegexScorer(r'\b\d+\b'), outputs=numeric_outputs)
print(report.summary())
for r in report.results:
    print(f'  {"✅" if r.passed else "❌"}  {r.output!r}')

## 4. RougeScorer (ROUGE-L)

Measures longest-common-subsequence overlap. Best for longer text outputs where exact match is too strict.

In [ ]:
from ag_evalforge import RougeScorer

summary_cases = EvalDataset(cases=[
    EvalCase(
        input='Summarise: The Eiffel Tower is a wrought-iron lattice tower in Paris, France.',
        expected='The Eiffel Tower is a lattice tower located in Paris.',
    ),
    EvalCase(
        input='Summarise: Python is a high-level, interpreted programming language.',
        expected='Python is an interpreted, high-level programming language.',
    ),
])

model_outputs = [
    'The Eiffel Tower is an iron lattice tower in Paris, France, built in 1889.',
    'Python is a scripting language.',
]

report = run_eval(summary_cases, RougeScorer(threshold=0.5), outputs=model_outputs)
print(report.summary())
for r in report.results:
    print(f'  {"✅" if r.passed else "❌"}  score={r.score:.3f}  output={r.output[:60]}')

## 5. LLMJudgeScorer — local judge using OpenAI

`LLMJudgeScorer` sends the (input, expected, output) triple to a judge model and asks it to score the output. Requires your own OpenAI key.

In [ ]:
# from ag_evalforge import LLMJudgeScorer
#
# judge = LLMJudgeScorer(
#     criteria='Is the response accurate, concise, and factually correct?',
#     model='gpt-4o-mini',
#     api_key=os.environ['OPENAI_API_KEY'],
#     threshold=0.7,
# )
#
# report = run_eval(cases, judge, outputs=outputs)
# print(report.summary())
# for r in report.results:
#     print(f'  {"✅" if r.passed else "❌"}  score={r.score:.2f}  reason={r.metadata.get("reason", "")[:60]}')

## 6. HostedScorer — ag hosted judge

`HostedScorer` uses the ag-evalforge API. With `scorer='llm_judge'` it two responses (A vs B) and returns which is preferred.

In [ ]:
from ag_evalforge import HostedScorer

# Direct scoring mode (0–1 score per response)
scorer = HostedScorer(api_key=AG_API_KEY, scorer='llm_judge')

direct_cases = EvalDataset(cases=[
    EvalCase(input='Explain recursion simply.', expected='A clear, simple explanation.'),
    EvalCase(input='What is a REST API?',       expected='An accurate API description.'),
])

direct_outputs = [
    'Recursion is when a function calls itself to solve a smaller version of the same problem.',
    'REST is an architectural style for building web services using HTTP methods.',
]

report = run_eval(direct_cases, scorer, outputs=direct_outputs)
print(report.summary())
for r in report.results:
    print(f'  score={r.score:.2f}  {r.output[:60]}')

## 7. Pairwise comparison — baseline vs candidate

In [ ]:
from ag_evalforge import compare

# Simulate two models with canned outputs
baseline_outputs  = ['Paris', '144', 'Shakespeare', '100', 'hola']
candidate_outputs = ['Paris, the city of light', '12 × 12 = 144', 'William Shakespeare', 'One hundred degrees', 'hola']

def baseline(prompt):  return baseline_outputs[cases.cases.index(next(c for c in cases.cases if c.input == prompt))]
def candidate(prompt): return candidate_outputs[cases.cases.index(next(c for c in cases.cases if c.input == prompt))]

base_report, cand_report = compare(cases, RougeScorer(threshold=0.6), baseline, candidate)

print('Baseline  :', base_report.summary())
print('Candidate :', cand_report.summary())

## 8. EvalReport — analysing results

In [ ]:
report = run_eval(cases, RougeScorer(threshold=0.5), outputs=outputs)

print('=== EvalReport ===')
print(f'Scorer      : {report.scorer_name}')
print(f'Total cases : {len(report.results)}')
print(f'Passed      : {sum(r.passed for r in report.results)}')
print(f'Failed      : {sum(not r.passed for r in report.results)}')
print(f'Pass rate   : {report.pass_rate:.1%}')
print(f'Mean score  : {report.mean_score:.3f}')
print()
print('Failed cases:')
for r in report.results:
    if not r.passed:
        print(f'  input={r.case.input!r}')
        print(f'  expected={r.case.expected!r}')
        print(f'  got={r.output!r}  score={r.score:.3f}')

## 9. Load a dataset from a file

`load_dataset` reads JSONL files with `{input, expected_output, metadata}` records.

In [ ]:
# from ag_evalforge import load_dataset
#
# dataset = load_dataset('eval_cases.jsonl')
# print(f'Loaded {len(dataset)} cases')
#
# # JSONL format:
# # {"input": "What is 2+2?", "expected_output": "4"}
# # {"input": "Capital of Germany?", "expected_output": "Berlin", "metadata": {"category": "geo"}}

## 10. Real-world: evaluating a QA system

**When to use ROUGE-L:** extractive QA, summarisation, and any task where
the model output is expected to share most of the same words as the reference.

**When NOT to use ROUGE-L:** paraphrase evaluation, open-ended generation, or
tasks where correct answers can vary widely in wording — use `LLMJudgeScorer`
or `HostedScorer(scorer='llm_judge')` instead.

In [ ]:
# ROUGE-L works well when outputs stay close to the reference wording.
# These outputs share most key words with the expected answers.

qa_cases = EvalDataset(cases=[
    EvalCase(input='What is the return policy?',          expected='Items can be returned within 30 days.'),
    EvalCase(input='How do I contact support?',           expected='Email support@example.com or call 1-800-555-0100.'),
    EvalCase(input='What payment methods do you accept?', expected='We accept Visa, Mastercard, PayPal, and bank transfer.'),
    EvalCase(input='When will my order arrive?',          expected='Standard delivery takes 3–5 business days.'),
])

# Model outputs that are close in wording to the expected answers
qa_outputs = [
    'Items can be returned within 30 days of purchase.',
    'Email support@example.com or call 1-800-555-0100 for support.',
    'We accept Visa, Mastercard, PayPal, and bank transfer.',
    'Standard delivery takes 3 to 5 business days.',
]

report = run_eval(qa_cases, RougeScorer(threshold=0.5), outputs=qa_outputs)

print('QA System Evaluation (ROUGE-L)')
print(f'Pass rate: {report.pass_rate:.1%}  |  Mean ROUGE-L: {report.mean_score:.3f}')
print()
for r in report.results:
    icon = '✅' if r.passed else '❌'
    print(f'{icon} [{r.score:.2f}] {r.case.input}')

In [ ]:
# For paraphrase-style outputs (same meaning, different words),
# use LLMJudgeScorer — it evaluates semantic correctness, not word overlap.

paraphrase_outputs = [
    'You can return products within 30 days of purchase.',          # different words, correct meaning
    'Contact us at support@example.com.',                          # omits phone — partial answer
    'We accept Visa and Mastercard.',                              # omits PayPal — partial answer
    'Delivery takes about 3 to 5 business days.',                  # correct paraphrase
]

rouge_report = run_eval(qa_cases, RougeScorer(threshold=0.5), outputs=paraphrase_outputs)
print(f'ROUGE-L pass rate : {rouge_report.pass_rate:.1%}  (penalises paraphrases)')

# Uncomment to compare with LLM judge (requires OPENAI_API_KEY):
# from ag_evalforge import LLMJudgeScorer
# llm_report = run_eval(qa_cases, LLMJudgeScorer(), outputs=paraphrase_outputs)
# print(f'LLM judge pass rate: {llm_report.pass_rate:.1%}  (rewards correct meaning)')